In [982]:
import pandas as pd
import numpy as np

# Default configuration for time periods in traffic data
input_file = "outputs/2025-11-10/COLA_sup_new_v7/model_run.csv"
run_folder = 'test_get_toll/COLA_sup_new_v7'

# Here we load th value of the counts and we multiply the peak hour values by a constant
lights_w = 1

heavies_w = 3
heavies_w_toll = 5
heavies_w_vot = 5

medium_A_w = 3 #1.5 # TBD: Maybe try 2.5 or 2.75 for every pce value
medium_A_w_toll = 5
medium_A_w_vot = 5

medium_B_w = 3 #2.75
medium_B_w_toll = 5
medium_B_w_vot = 5

heavy_A_w = 3 # 2.75
heavy_A_w_toll = 5
heavy_A_w_vot = 5

heavy_B_w = 3
heavy_B_w_toll = 5
heavy_B_w_vot = 5

traffic_condition = 1500
speed_condition = 55

# Default time periods list (for reference)
default_time_periods = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

# Create the base scenario: hour -> time period mapping
hour_to_period = {
    0: "Night",
    1: "Night",
    2: "Night",
    3: "Night",
    4: "Night",
    5: "Night",
    6: "AM-Early",
    7: "AM-Peak",
    8: "AM-Peak",
    9: "AM-Shoulder",
    10: "MD",
    11: "MD",
    12: "MD",
    13: "MD",
    14: "MD",
    15: "PM-Shoulder",
    16: "PM-Peak",
    17: "PM-Peak",
    18: "PM-Peak",
    19: "PM-Late",
    20: "PM-Late",
    21: "PM-Late",
    22: "Night",
    23: "Night"
}

period_to_period = {
    'Evening': 'Night',
    'Evening': 'PM-Late',
    'EarlyAM': 'AM-Early',
    'AM': 'AM-Peak',
    'AM': 'AM-Shoulder',
    'Midday': 'MD',
    'Midday': 'PM-Shoulder',
    'PM': 'PM-Peak'
}

# Define the segments and their parameters

awt_adt = 1.1 # Average weekday traffic (AWT) to average daily traffic (ADT) ratio
peak_factor = 1 # 1.05 # Peak factor for adjustment at peak hour traffic

hov_percentage = pd.DataFrame({
    'Year' : [2025,2032,2040,2050],
    'HOV percentage' : [0,0,0,0]
})

hov_percentage.set_index('Year', inplace=True)

# Define segment parameters base
# seg_params = pd.DataFrame({
#     'SegDir':   ["1WB","1EB","2WB","2EB","3WB","3EB","4WB","4EB","5WB","5EB","6WB","6EB","7WB","7EB","8WB","8EB","9WB","9EB","10WB","10EB","11WB","11EB","12WB","12EB","13WB","13EB"],
#     'Length':    [2.4,2.4,3.2,3.2,3.2,3.2,1.8,1.8,4.2,4.2,2.0,2.0,1.6,1.6,3.9,3.9,3.5,3.5,3.5,3.5,1,1,3.5,3.5,1.8,1.8], # 1.5 miles in S13 instead of s5 and replacing the 3.3 from the extension miles # TBD: I added 1.5 miles to S5, check if it's correct
#     'Inscope':   [0.45,0.45,0.534,0.534,0.466,0.466,0.573,0.573,0.801,0.801,0.65,0.65,0.65,0.65,0.65,0.65,0.65,0.65,0.621,0.621,0.65,0.65,0.65,0.65,0.65,0.65], # TBD: compute values per segment
#     'Lanes_GP':  [5]*8 + [4]*10 + [5]*4 + [3]*2 + [5]*2, #
#     'Lanes_ML':  [2]*26, # Lanes_ML': [2,2,2,2,2,2,2,2,3,3,2,2,2,2], # Do test changing segment 5
#     'CapPerLane_GP': [2000]*26,
#     'CapPerLane_ML': [1800]*26,
#     'Speed_GP':  [65]*22 + [55]*2 + [65]*2,
#     'Speed_ML':  [70]*26,
#     'Alpha_GP':  [1]*26,
#     'Beta_GP':   [6]*26,
#     'Alpha_ML':  [1]*26,
#     'Beta_ML':   [6]*26,
#     'Min_Toll_2016': [None]*26,
#     'Max_Toll_2016': [None]*26,
#     'LanesGP_AM_Peak': [5]*26,
#     'LanesGP_PM_Peak': [5]*26,
# })

# With phase 2

# lengths = [2.4,2.4,3.2,3.2,3.2,3.2,1.8,1.8,3.3,6.1,2.0,2.0,1.6,1.6,3.9,3.9,3.5,3.5,3.5,3.5,1,1,3.5,3.5,1.8,1.8]

# # lengths = [1.94,3.36,2.66,4.48,3.2,3.2,1.8,1.8,3.3,6.1,2.0,2.0,1.6,1.6,3.9,3.9,3.5,3.5,2.69,4.5,1,1,3.5,3.5,1.8,1.8]

# # Define segment parameters base
# seg_params = pd.DataFrame({
#     'SegDir':   ["1WB","1EB","2WB","2EB","3WB","3EB","4WB","4EB","5WB","5EB","6WB","6EB","7WB","7EB","8WB","8EB","9WB","9EB","10WB","10EB","11WB","11EB","12WB","12EB","13WB","13EB"],
#     'Length':    lengths, # 1.5 miles in S13 instead of s5 and replacing the 3.3 from the extension miles # TBD: I added 1.5 miles to S5, check if it's correct
#     'Inscope':   [0.641,0.641,0.756,0.756,0.466,0.466,0.573,0.573,0.801,0.801,0.65,0.65,0.65,0.65,0.65,0.65,0.65,0.65,0.793,0.793,0.65,0.65,0.65,0.65,0.65,0.65], # TBD: compute values per segment
#     'Lanes_GP':  [5]*8 + [4]*10 + [5]*4 + [3]*2 + [5]*2, #
#     'Lanes_ML':  [2]*26, # Lanes_ML': [2,2,2,2,2,2,2,2,3,3,2,2,2,2], # Do test changing segment 5
#     'CapPerLane_GP': [2000]*26,
#     'CapPerLane_ML': [1800]*26,
#     'Speed_GP':  [65]*22 + [55]*2 + [65]*2,
#     'Speed_ML':  [70]*26,
#     'Alpha_GP':  [1]*26,
#     'Beta_GP':   [6]*26,
#     'Alpha_ML':  [1]*26,
#     'Beta_ML':   [6]*26,
#     'Min_Toll_2016': [None]*26,
#     'Max_Toll_2016': [None]*26,
#     'LanesGP_AM_Peak': [5]*26,
#     'LanesGP_PM_Peak': [5]*26,
# })

# COLA
lengths = [2.4,2.4,3.2,3.2,3.2,3.2,1.8,1.8,4.2,4.2,2.0,2.0,1.6,1.6,3.9,3.9,3.5,3.5,3.5,3.5,1,1,3.5,3.5,1.8,1.8]

# lengths = [1.94,3.36,2.66,4.48,3.2,3.2,1.8,1.8,3.3,6.1,2.0,2.0,1.6,1.6,3.9,3.9,3.5,3.5,2.69,4.5,1,1,3.5,3.5,1.8,1.8]

# inscope = [0.85,0.85,0.868,0.868,0.863,0.863,0.95,0.95,0.861,0.861,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.87,0.87] + [0.83]*6

inscope = [0.85,0.85,0.868,0.868,0.863,0.863,0.95,0.95,0.861,0.861,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.87,0.87] + [0.83]*2 + [0.8]*2 + [0.83]*2

# Define segment parameters base
seg_params = pd.DataFrame({
    'SegDir':   ["1WB","1EB","2WB","2EB","3WB","3EB","4WB","4EB","5WB","5EB","6WB","6EB","7WB","7EB","8WB","8EB","9WB","9EB","10WB","10EB","11WB","11EB","12WB","12EB","13WB","13EB"],
    'Length':    lengths, # 1.5 miles in S13 instead of s5 and replacing the 3.3 from the extension miles # TBD: I added 1.5 miles to S5, check if it's correct
    'Inscope':   inscope, # TBD: compute values per segment
    'Lanes_GP':  [5]*8 + [4]*10 + [5]*4 + [3]*2 + [5]*2, #
    'Lanes_ML':  [2]*26, # Lanes_ML': [2,2,2,2,2,2,2,2,3,3,2,2,2,2], # Do test changing segment 5
    'CapPerLane_GP': [2000]*26,
    'CapPerLane_ML': [1800]*26,
    'Speed_GP':  [65]*22 + [55]*2 + [65]*2,
    'Speed_ML':  [70]*26,
    'Alpha_GP':  [1]*26,
    'Beta_GP':   [6]*26,
    'Alpha_ML':  [1]*26,
    'Beta_ML':   [6]*26,
    'Min_Toll_2016': [None]*26,
    'Max_Toll_2016': [None]*26,
    'LanesGP_AM_Peak': [5]*26,
    'LanesGP_PM_Peak': [5]*26,
})


seg_params.set_index('SegDir', inplace=True)

# Compute capacities as lanes * cap per lane
seg_params['Cap_GP'] = seg_params['Lanes_GP'] * seg_params['CapPerLane_GP']
seg_params['Cap_ML'] = seg_params['Lanes_ML'] * seg_params['CapPerLane_ML']

# Compute peak capacities as Alpha * base capacity
seg_params['CapGP_Peak'] = seg_params['Alpha_GP'] * seg_params['Cap_GP']
seg_params['CapML_Peak'] = seg_params['Alpha_ML'] * seg_params['Cap_ML']

# Optional: if you want integer capacities
seg_params[['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']] = seg_params[
    ['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']
].astype(int)

# Preview
seg_params

,Length,Inscope,Lanes_GP,Lanes_ML,CapPerLane_GP,CapPerLane_ML,Speed_GP,Speed_ML,Alpha_GP,Beta_GP,Alpha_ML,Beta_ML,Min_Toll_2016,Max_Toll_2016,LanesGP_AM_Peak,LanesGP_PM_Peak,Cap_GP,Cap_ML,CapGP_Peak,CapML_Peak
SegDir,,,,,,,,,,,,,,,,,,,,
1WB,2.4,0.850,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
1EB,2.4,0.850,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
2WB,3.2,0.868,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
2EB,3.2,0.868,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
3WB,3.2,0.863,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
3EB,3.2,0.863,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
4WB,1.8,0.950,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
4EB,1.8,0.950,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
5WB,4.2,0.861,4,2,2000,1800,65,70,1,6,1,6,None,None,5,5,8000,3600,8000,3600


In [983]:
lookup_period_file = r"inputs/LookUp_Period.csv"

lookup_period = pd.read_csv(
    lookup_period_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

# Clip y reasignar
lookup_period = lookup_period * 0

lookup_period


,Bonus/Mile,4 Periods
Period,,
Night,0.0,
AM-Early,0.0,
AM-Peak,0.0,
AM-Shoulder,0.0,
MD,0.0,
PM-Shoulder,0.0,
PM-Peak,0.0,
PM-Late,0.0,


In [984]:
def get_vot(row):

    max_VC = 1.2  # TBD: check if we need to change this value
    ETC_discount = 0.15
    
    captureRateLights =  row["CaptureRateLights"]

    captureRateMediumA =  row["CaptureRateMediumA"]

    captureRateMediumB =  row["CaptureRateMediumB"]

    tollLights = row["TollLights"]

    tollMediumA = tollLights * medium_A_w_toll

    tollMediumB = tollLights * medium_B_w_toll

    ml_pce = (
        row["InScopeLights"] * lights_w * captureRateLights + 
        row["InScopeMediumA"] * medium_A_w * captureRateMediumA +
        row["InScopeMediumB"] * medium_B_w * captureRateMediumB
    )

    gp_pce = np.min(np.array([row["Corridor PCE"] - ml_pce, row["Max VC"] * seg_params['Cap_GP'][row['SegDir']]]))
    
    speedML = row["Speed ML"] / (1 + row["Alpha ML"] * (((ml_pce + row["HOV3"]) / row["Capacity ML"])** row["Beta ML"])) # TBD: maintain beta ML for future tests

    timeML = 60 * row["Length"] / speedML

    speedGP =  row["Speed GP"] / (1 + row["Alpha GP"] * ((gp_pce / row["Capacity GP"]) ** row["Beta GP"]))

    timeGP = 60 * row["Length"] / speedGP

    timeSavings = timeML - timeGP

    gp_vc = gp_pce / row["Capacity GP"]

    ml_vc = (ml_pce + row["HOV3"]) / row["Capacity ML"]

    bonusRel = 1.37 * 60 * row["Length"] * ((1/speedGP)-(1/row["Speed GP"])) 

    bonusAux = (2.5 if (row["Period"] == "AM-Peak" or row["Period"] == "PM-Peak") else 1.6) 

    bonusSeg = (lookup_period.loc[row["Period"], "Bonus/Mile"] + (bonusAux if gp_vc > 0.3 else 0) * (gp_vc - ml_vc)) * row["Length"] # TBD: add condition for gp_vc

    votLights = -60 * (1 - ETC_discount) * row["Length"] * tollLights / (timeSavings - bonusSeg - bonusRel) # TBD: add household income effect

    votMediumA = -60 * (1 - ETC_discount) * row["Length"] * tollMediumA / (timeSavings - bonusSeg - bonusRel)

    votMediumB = -60 * (1 - ETC_discount) * row["Length"] * tollMediumB / (timeSavings - bonusSeg - bonusRel)

    return pd.Series([votLights, votMediumA, votMediumB, speedGP, speedML, bonusSeg, timeML, timeGP, timeSavings, bonusRel], index=["VOT Lights", "VOT Medium A", "VOT Medium B", "Speed GP", "Speed ML", "Bounus Seg", "Time ML", "Time GP", "Time Savings", "BonusRel"])

In [985]:
from scipy.stats import lognorm
from scipy.optimize import minimize, least_squares, Bounds
import numpy as np

ETC_discount = 0.15
max_VC = 1.2  # TBD: check if we need to change this value

def objective_integrated(x, row, tollLights):

    captureRateLights, captureRateMediumA, captureRateMediumB = x

    tollMediumA = tollLights * medium_A_w_toll
    tollMediumB = tollLights * medium_B_w_toll
    tollHeavyA = tollLights * heavy_A_w_toll

    ml_pce = (
        row["InScopeLights"] * lights_w * captureRateLights + 
        row["InScopeMediumA"] * medium_A_w * captureRateMediumA +
        row["InScopeMediumB"] * medium_B_w * captureRateMediumB
    )

    gp_pce = np.min(np.array([row["Corridor PCE"] - ml_pce, row["Max VC"] * seg_params['Cap_GP'][row['SegDir']]]))
    
    speedML = row["Speed ML"] / (1 + row["Alpha ML"] * (((ml_pce + row["HOV3"]) / row["Capacity ML"])** row["Beta ML"])) # TBD: maintain beta ML for future tests

    timeML = 60 * row["Length"] / speedML

    speedGP =  row["Speed GP"] / (1 + row["Alpha GP"] * ((gp_pce / row["Capacity GP"]) ** row["Beta GP"]))

    timeGP = 60 * row["Length"] / speedGP

    timeSavings = timeML - timeGP

    gp_vc = gp_pce / row["Capacity GP"]

    ml_vc = (ml_pce + row["HOV3"]) / row["Capacity ML"]

    bonusRel = 1.37 * 60 * row["Length"] * ((1/speedGP)-(1/row["Speed GP"]))

    bonusAux = (2.5 if (row["Period"] == "AM-Peak" or row["Period"] == "PM-Peak") else 1.6)

    bonusSeg = (lookup_period.loc[row["Period"], "Bonus/Mile"] + (bonusAux if gp_vc > 0.3 else 0) * (gp_vc - ml_vc)) * row["Length"] # TBD: add condition for gp_vc

    votLights = -60 * (1 - ETC_discount) * row["Length"] * tollLights / (timeSavings - bonusSeg - bonusRel) # TBD: add household income effect

    votMediumA = -60 * (1 - ETC_discount) * row["Length"] * tollMediumA / (timeSavings - bonusSeg - bonusRel)

    votMediumB = -60 * (1 - ETC_discount) * row["Length"] * tollMediumB / (timeSavings - bonusSeg - bonusRel)

    # TBD: Integrate Reliability
    # Here we compute the lognormal cumulative function for the light vehicles

    mu_lights = row["B1"]

    mu_heavies = mu_lights

    std_dev = row["B2"]

    scale_lights = np.exp(mu_lights)

    scale_mediumA = np.exp(mu_heavies) * medium_A_w_vot # TBD: change heavies betas (ask Borja)

    scale_mediumB = np.exp(mu_heavies) * medium_B_w_vot # TBD: change heavies betas (ask Borja)

    # We create a lognormal distribution object

    dist_lights = lognorm(s=std_dev, scale=scale_lights)

    dist_mediumA = lognorm(s=std_dev, scale=scale_mediumA)

    dist_mediumB = lognorm(s=std_dev, scale=scale_mediumB)

    calcCaptureRateLights = 1 - dist_lights.cdf(votLights)

    calcCaptureMediumA = 1 - dist_mediumA.cdf(votMediumA)

    calcCaptureMediumB = 1 - dist_mediumB.cdf(votMediumB)

    convergenceLights = calcCaptureRateLights - captureRateLights

    convergenceMediumA = calcCaptureMediumA - captureRateMediumA

    convergenceMediumB = calcCaptureMediumB - captureRateMediumB

    return convergenceLights ** 2 + convergenceMediumA ** 2 + convergenceMediumB ** 2

def optimize_capture(row, M, toll_value):

    capture_first_guess = row["CaptureRateLights"]
    max_capture = row["MaxCapture"]

    ub = [
        max_capture,
        max_capture,
        max_capture
    ]

    lb = [
        0,
        0,
        0
    ]

    bounds = Bounds(lb=lb, ub=ub)

    x0 = [
        capture_first_guess,
        capture_first_guess,
        capture_first_guess
    ] # Initial guess, TBD

    Max_iter = 10000000

    # Solve
    result = minimize(
        objective_integrated,
        x0,
        method='trust-constr',
        args=(row,toll_value),
        bounds=bounds,
        # constraints=[nlc],
        options={
            "verbose": 0,
            "maxiter": Max_iter,
            "gtol": 1e-6,
            "xtol": 1e-6,
            "barrier_tol": 1e-6,
            "initial_tr_radius": 1.0,
            "initial_constr_penalty": 1.0,
            "sparse_jacobian": True
        }
    )

    x_opt = result.x
    z_opt = result.fun
    x_opt = np.minimum(bounds.ub, np.maximum(bounds.lb, result.x))
    return x_opt

In [986]:
import pandas as pd
import sys

first_model_df = pd.read_csv(f"{input_file}")


# first_model_df = first_model_df[(first_model_df["Year"] == 2050) or ()].reset_index()

def optimize_row(row):

    # We get the value of the toll for the highest value of revenue

    captureLights = row["CaptureRateLights"]

    captureMediumA = row["CaptureRateMediumA"]

    captureMediumB = row["CaptureRateMediumB"]

    ml_pce = (row["InScopeLights"] * captureLights + row["HOV3"] + 
              row["InScopeMediumA"] *  captureMediumA * medium_A_w + 
              row["InScopeMediumB"] *  captureMediumB * medium_B_w)
    
    traffic_lane =  ml_pce/ seg_params.loc[row["SegDir"], 'Lanes_ML']

    pce_speed = seg_params.loc[row["SegDir"], 'Inscope'] * (row["TotalLights"] * lights_w * captureLights + 
                                                            row["TotalMediumA"] * medium_A_w * captureMediumA +
                                                            row["TotalMediumB"] * medium_B_w * captureMediumB
                                                            )

    speedML = row["Speed ML"] / (1 + row["Alpha ML"] * (((ml_pce + row["HOV3"]) / row["Capacity ML"])** row["Beta ML"])) # TBD: maintain beta ML for future tests

    toll = row["TollLights"]

    changed = False


    # print(f'Segment: {row["SegDir"]}, Period: {row["Period"]}, toll: {toll}, traffic: {traffic_lane}, speed: {speedML}')

    while (traffic_lane > 1500) and (speedML < 45):
        changed = True
        x_opt = optimize_capture(row, 1, toll)
        captureLights, captureMediumA, captureMediumB = x_opt
        ml_pce = (row["InScopeLights"] * captureLights + row["HOV3"] + 
            row["InScopeMediumA"] *  captureMediumA * medium_A_w + 
            row["InScopeMediumB"] *  captureMediumB * medium_B_w
            )

        traffic_lane =  ml_pce/ seg_params.loc[row["SegDir"], 'Lanes_ML']
        speedML = row["Speed ML"] / (1 + row["Alpha ML"] * (((ml_pce + row["HOV3"]) / row["Capacity ML"])** row["Beta ML"])) # TBD: maintain beta ML for future tests
        # print(f'Segment: {row["SegDir"]}, Period: {row["Period"]}, toll: {toll}, traffic: {traffic_lane}, speed: {speedML}')
        toll += 0.1
        if toll > 13.6:
            break

    if changed:
        changed = False
        print(f'Segment: {row["SegDir"]}, Period: {row["Period"]}, toll: {toll}, traffic: {traffic_lane}, speed: {speedML}')

    return pd.Series([captureLights, captureMediumA, captureMediumB, toll], 
                index=["CaptureRateLights", "CaptureRateMediumA", "CaptureRateMediumB", "TollLights"])

first_model_df[["CaptureRateLights", "CaptureRateMediumA", "CaptureRateMediumB", "TollLights"]] = first_model_df.apply(
    optimize_row, axis=1, result_type='expand'
)

C:\Users\crisp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\scipy\optimize\_differentiable_functions.py:376: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)


Segment: 10WB, Period: PM-Shoulder, toll: 4.71999266181115, traffic: 1129.8084388865725, speed: 65.96619159152873
Segment: 10WB, Period: PM-Peak, toll: 4.719992579581833, traffic: 1389.4800448665721, speed: 57.775698376038584
Segment: 10EB, Period: AM-Peak, toll: 4.719992262334293, traffic: 1029.7372885265247, speed: 67.62939545870555
Segment: 11WB, Period: PM-Shoulder, toll: 4.719971896096129, traffic: 973.8952751405543, speed: 68.2869261468916
Segment: 11WB, Period: PM-Peak, toll: 4.719973841537936, traffic: 1231.0897448313478, speed: 63.50049392172854
Segment: 11EB, Period: AM-Peak, toll: 4.719978670988158, traffic: 884.9901342061861, speed: 69.02500587864832
Segment: 13WB, Period: AM-Peak, toll: 4.719992400938112, traffic: 945.5444786849204, speed: 68.55946385412675
Segment: 1WB, Period: PM-Peak, toll: 10.409989578671425, traffic: 1630.5369956899617, speed: 45.08795240952129
Segment: 1EB, Period: AM-Peak, toll: 11.209999996148603, traffic: 1630.8934748819918, speed: 45.066901966378

In [987]:
first_model_df["TollMediumA"] = first_model_df.apply(
    lambda row: row["TollLights"] * medium_A_w_toll,
    axis=1
)

first_model_df["TollMediumB"] = first_model_df.apply(
    lambda row: row["TollLights"] * medium_B_w_toll,
    axis=1
)

first_model_df[["VOT Lights", "VOT MediumA", "VOT MediumB", "Speed GP Real", "Speed ML Real", "Bounus Seg", "Time ML", "Time GP", "Time Savings", "BonusRel"]] = first_model_df.apply(
    get_vot, axis=1, result_type='expand'
)

first_model_df

,Unnamed: 0,Year,SegDir,Segment,Direction,Period,Hours/Day,Peak,4Periods,Length,...,TransactionsDay_Lights,TransactionsDay_MediumA,TransactionsDay_MediumB,TransactionsDay,TransactionsLightsLength,TransactionsMediumALength,TransactionsMediumBLength,TotalVeh_hours,Difference in Travel Time,Implied Min VOT
0,0,2025,1WB,1,WB,Night,8,OP,NT,2.4,...,1431.199733,28.000029,1.600064,1460.799826,3434.879360,67.200069,3.840153,12944.000000,-0.072105,75.067306
1,1,2025,1WB,1,WB,AM-Early,1,OP,AM,2.4,...,569.178332,26.463197,1.321979,596.963508,1366.027997,63.511672,3.172751,5734.000000,-0.104727,390.579643
2,2,2025,1WB,1,WB,AM-Peak,2,Peak,AM,2.4,...,2526.182358,97.305452,13.675364,2637.163174,6062.837658,233.533086,32.820874,15265.908890,-0.196811,238.315241
3,3,2025,1WB,1,WB,AM-Shoulder,1,OP,AM,2.4,...,784.503654,27.755124,5.492565,817.751343,1882.808770,66.612299,13.182156,6479.000000,-0.180664,265.812807
4,4,2025,1WB,1,WB,MD,5,OP,MD,2.4,...,5138.939523,186.054632,35.500551,5360.494705,12333.454856,446.531116,85.201321,34525.000000,-0.229596,196.493909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,827,2050,13EB,13,EB,AM-Shoulder,1,OP,AM,1.8,...,1936.156625,89.082683,18.589869,2043.829177,3485.081926,160.348829,33.461764,8008.800000,-0.409484,174.152207
636,828,2050,13EB,13,EB,MD,5,OP,MD,1.8,...,12595.871379,398.523278,73.038887,13067.433543,22672.568481,717.341901,131.469996,50881.500000,-0.563351,162.501727
637,829,2050,13EB,13,EB,PM-Shoulder,1,OP,PM,1.8,...,3601.041489,75.001700,6.179508,3682.222697,6481.874680,135.003060,11.123114,10452.100000,-0.307524,232.367763
638,830,2050,13EB,13,EB,PM-Peak,3,Peak,PM,1.8,...,11354.910309,243.054179,12.701092,11610.665580,20438.838556,437.497523,22.861966,34217.163458,-0.199808,342.159693


In [988]:
first_model_df["MLVeh_Lights"] = first_model_df.apply(
    lambda row: row["InScopeLights"] * row["CaptureRateLights"],
    axis=1
)

first_model_df["GPVeh_Lights"] = first_model_df.apply(
    lambda row: row["TotalLights"] - row["MLVeh_Lights"] - row["HOV3"],
    axis=1
)

first_model_df["MLVeh_MediumA"] = first_model_df.apply(
    lambda row: row["InScopeMediumA"] * row["CaptureRateMediumA"],
    axis=1
)

first_model_df["GPVeh_MediumA"] = first_model_df.apply(
    lambda row: row["TotalMediumA"] - row["MLVeh_MediumA"],
    axis=1
)

first_model_df["MLVeh_MediumB"] = first_model_df.apply(
    lambda row: row["InScopeMediumB"] * row["CaptureRateMediumB"],
    axis=1
)

first_model_df["GPVeh_MediumB"] = first_model_df.apply(
    lambda row: row["TotalMediumB"] - row["MLVeh_MediumB"],
    axis=1
)

first_model_df["MLVeh"] = first_model_df.apply(
    lambda row: row["MLVeh_Lights"] + row["MLVeh_MediumA"] + row["MLVeh_MediumB"],
    axis=1
)

first_model_df["GPVeh"] = first_model_df.apply(
    lambda row: row["GPVeh_Lights"] + row["GPVeh_MediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["GPVehDay"] = first_model_df.apply(
    lambda row: row["GPVeh"] * row["Hours/Day"],
    axis=1
)

first_model_df["ML PCE"] = first_model_df.apply(
    lambda row:row["MLVeh_Lights"] * lights_w
    + row["MLVeh_MediumA"] * medium_A_w
    + row["MLVeh_MediumB"] * medium_B_w,
    axis=1
)

first_model_df["ML V/C"] = first_model_df.apply(
    lambda row:row["ML PCE"] / row["Capacity ML"],
    axis=1
)

first_model_df["ML Volume"] = first_model_df.apply(
    lambda row: row["ML PCE"] * row["Hours/Day"],
    axis=1
)

first_model_df["ML PCE Total"] = first_model_df.apply(
    lambda row: row["ML PCE"] * row["Hours/Day"] * row["Length"],
    axis=1
)

first_model_df["GP PCE"] = first_model_df.apply(
    lambda row:row["GPVeh_Lights"] * lights_w
    + row["GPVeh_MediumA"] * medium_A_w
    + row["TotalMediumB"] * medium_B_w
    + row["TotalHeavyA"] * heavy_A_w
    + row["TotalHeavyB"] * heavy_B_w, # TBD: is HOV included in the PCE?
    axis=1
)

first_model_df["GP Volume"] = first_model_df.apply(
    lambda row: row["GP PCE"] * row["Hours/Day"],
    axis=1
)

first_model_df["GP PCE Total"] = first_model_df.apply(
    lambda row: row["GP PCE"] * row["Hours/Day"] * row["Length"],
    axis=1
)

# Corridor Vals

first_model_df["Highway PCE"] = first_model_df.apply(
    lambda row: row["ML PCE"] + row["GP PCE"],
    axis=1
)

first_model_df["Highway Volume"] = first_model_df.apply(
    lambda row: row["ML Volume"] + row["GP Volume"],
    axis=1
)

first_model_df["Highway PCE Total"] = first_model_df.apply(
    lambda row: row["ML PCE Total"] + row["GP PCE Total"],
    axis=1
)

first_model_df["Highway V/C"] = first_model_df.apply(
    lambda row: (row["ML PCE"] + row["GP PCE"]) / (row["Capacity GP"]),
    axis=1
)

# We add the TollPerSeg column

first_model_df["TollLightsPerSeg"] = first_model_df.apply(
    lambda row: row["TollLights"] * row["Length"],
    axis=1
)

first_model_df["TollMediumAPerSeg"] = first_model_df.apply(
    lambda row: row["TollMediumA"] * row["Length"],
    axis=1
)

first_model_df["TollMediumBPerSeg"] = first_model_df.apply(
    lambda row: row["TollMediumB"] * row["Length"],
    axis=1
)

# Toll blend
first_model_df["TollBlend"] = first_model_df.apply(
    lambda row: (row["TollLights"] * row["MLVeh_Lights"] 
                + row["TollMediumA"] * row["MLVeh_MediumA"]
                +row["TollMediumB"] * row["MLVeh_MediumB"]
                ) / (row["MLVeh"]),
    axis=1
)

first_model_df["RevenuePerHour"] = first_model_df.apply(
    lambda row: row["TollLightsPerSeg"] * row["MLVeh_Lights"]
    + row["TollMediumAPerSeg"] * row["MLVeh_MediumA"]
    + row["TollMediumBPerSeg"] * row["MLVeh_MediumB"],
    axis=1
)

first_model_df["RevenuePerHourLights"] = first_model_df.apply(
    lambda row: row["TollLightsPerSeg"] * row["MLVeh_Lights"],
    axis=1
)

first_model_df["RevenuePerHourMediumA"] = first_model_df.apply(
    lambda row: row["TollMediumAPerSeg"] * row["MLVeh_MediumA"],
    axis=1
)

first_model_df["RevenuePerHourMediumB"] = first_model_df.apply(
    lambda row: row["TollMediumBPerSeg"] * row["MLVeh_MediumB"],
    axis=1
)

first_model_df["RevenuePerDay"] = first_model_df.apply(
    lambda row: row["RevenuePerHour"] * row["Hours/Day"],
    axis=1
)

first_model_df["RevenuePerDayLights"] = first_model_df.apply(
    lambda row: row["RevenuePerHourLights"] * row["Hours/Day"],
    axis=1
)

first_model_df["RevenuePerDayMediumA"] = first_model_df.apply(
    lambda row: row["RevenuePerHourMediumA"] * row["Hours/Day"],
    axis=1
)

first_model_df["RevenuePerDayMediumB"] = first_model_df.apply(
    lambda row: row["RevenuePerHourMediumB"] * row["Hours/Day"],
    axis=1
)

first_model_df["TransactionsDay_Lights"] = first_model_df.apply(
    lambda row: row["Hours/Day"] * row["MLVeh_Lights"],
    axis=1
)

first_model_df["TransactionsDay_MediumA"] = first_model_df.apply(
    lambda row: row["Hours/Day"] * row["MLVeh_MediumA"],
    axis=1
)

first_model_df["TransactionsDay_MediumB"] = first_model_df.apply(
    lambda row: row["Hours/Day"] * row["MLVeh_MediumB"],
    axis=1
)

first_model_df["TransactionsDay"] = first_model_df.apply(
    lambda row: row["TransactionsDay_Lights"] + row["TransactionsDay_MediumA"] + row["TransactionsDay_MediumB"],
    axis=1
)

first_model_df["TransactionsLightsLength"] = first_model_df.apply(
    lambda row: row["TransactionsDay_Lights"] * row["Length"],
    axis=1
)

first_model_df["TransactionsMediumALength"] = first_model_df.apply(
    lambda row: row["TransactionsDay_MediumA"] * row["Length"],
    axis=1
)

first_model_df["TransactionsMediumBLength"] = first_model_df.apply(
    lambda row: row["TransactionsDay_MediumB"] * row["Length"],
    axis=1
)

first_model_df["TotalVeh_hours"] = first_model_df.apply(
    lambda row: row["Hours/Day"] * row["TotalVeh"],
    axis=1
)

# Extra ones for the pivot

first_model_df["Difference in Travel Time"] = first_model_df.apply(
    lambda row: row["Time Savings"] / row["Time GP"],
    axis=1
)

first_model_df["Implied Min VOT"] = first_model_df.apply(
    lambda row: 60 * row["TollLights"] / (row["Time GP"] - row["Time ML"]),
    axis=1
)

first_model_df.to_csv(f'{run_folder}/test_model_run.csv')

first_model_df

,Unnamed: 0,Year,SegDir,Segment,Direction,Period,Hours/Day,Peak,4Periods,Length,...,TransactionsDay_Lights,TransactionsDay_MediumA,TransactionsDay_MediumB,TransactionsDay,TransactionsLightsLength,TransactionsMediumALength,TransactionsMediumBLength,TotalVeh_hours,Difference in Travel Time,Implied Min VOT
0,0,2025,1WB,1,WB,Night,8,OP,NT,2.4,...,1431.199733,28.000029,1.600064,1460.799826,3434.879360,67.200069,3.840153,12944.000000,-0.072105,75.067306
1,1,2025,1WB,1,WB,AM-Early,1,OP,AM,2.4,...,569.178332,26.463197,1.321979,596.963508,1366.027997,63.511672,3.172751,5734.000000,-0.104727,390.579643
2,2,2025,1WB,1,WB,AM-Peak,2,Peak,AM,2.4,...,2526.182358,97.305452,13.675364,2637.163174,6062.837658,233.533086,32.820874,15265.908890,-0.196811,238.315241
3,3,2025,1WB,1,WB,AM-Shoulder,1,OP,AM,2.4,...,784.503654,27.755124,5.492565,817.751343,1882.808770,66.612299,13.182156,6479.000000,-0.180664,265.812807
4,4,2025,1WB,1,WB,MD,5,OP,MD,2.4,...,5138.939523,186.054632,35.500551,5360.494705,12333.454856,446.531116,85.201321,34525.000000,-0.229596,196.493909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,827,2050,13EB,13,EB,AM-Shoulder,1,OP,AM,1.8,...,1936.156625,89.082683,18.589869,2043.829177,3485.081926,160.348829,33.461764,8008.800000,-0.409484,174.152207
636,828,2050,13EB,13,EB,MD,5,OP,MD,1.8,...,12595.871379,398.523278,73.038887,13067.433543,22672.568481,717.341901,131.469996,50881.500000,-0.563351,162.501727
637,829,2050,13EB,13,EB,PM-Shoulder,1,OP,PM,1.8,...,3097.766187,64.523321,5.318523,3167.608031,5575.979136,116.141977,9.573342,10452.100000,-0.664529,167.850642
638,830,2050,13EB,13,EB,PM-Peak,3,Peak,PM,1.8,...,10000.047708,213.888450,11.182540,10225.118698,18000.085874,384.999209,20.128573,34217.163458,-0.596478,186.355785


In [989]:
period_order = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

first_model_df["Period"] = pd.Categorical(
    first_model_df["Period"],
    categories=period_order,
    ordered=True
)

first_model_df["GP V/C normal"] = first_model_df.apply(
    lambda row: row["GP PCE"] / seg_params.loc[row["SegDir"], 'Cap_GP'],
    axis=1
)

first_model_df["GP V/C capacity factors"] = first_model_df.apply(
    lambda row: row["GP PCE"] / row["Capacity GP"],
    axis=1
)

first_model_df["CaptureRateLights"] = first_model_df.apply(
    lambda row: row["CaptureRateLights"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

In [990]:
# Define a light green style
def apply_light_green_style(df):
    return (df.style
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', "#7afc7f"),  # Light green
            ('color', '000000'),
            ('font-weight', 'bold'),
            ('border', '1px solid black')
        ]
    }])
    .set_properties(**{'border': '1px solid black'})  # Optional: add borders to data cells
)

In [991]:
# model_file = 'outputs/2025-10-13/base_gdt_v2/model_run.csv'

# first_model_df = pd.read_csv(model_file)

# First Column

def generate_pivot(df, pivot_file, years):

    with pd.ExcelWriter(pivot_file, engine='openpyxl') as writer:

        for year_val in years:

            df_filtered = df[df["Year"] == year_val]

            lengths = df_filtered.pivot_table(
                index="Period",  # Single row index
                columns=["Direction", "SegDir"],
                values="Length",
                aggfunc='first'  # Takes the first value (since they're all the same)
            ).round(2)

            lengths = df_filtered[['Segment', 'Length']].drop_duplicates().set_index("Segment")

            lengths = lengths.T

            sumTollLights = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TollLights"
            ).round(2)

            sumCaptureRateLights = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="CaptureRateLights"
            ).round(2)

            revenuePerPeriod = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="RevenuePerDay"
            ).round(2)

            revenuePerPeriod_LV = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="RevenuePerDayLights"
            ).round(2)

            revenuePerPeriod_MediumA = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="RevenuePerDayMediumA"
            ).round(2)

            revenuePerPeriod_MediumB = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="RevenuePerDayMediumB"
            ).round(2)

            percentage_revenue = revenuePerPeriod / revenuePerPeriod.sum()

            # Column 2

            ML_PCE_miles = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="ML PCE Total"
            ).round(2)

            ML_Volume = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="ML Volume"
            ).round(2)

            ML_flow = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="ML PCE"
            ).round(2)

            ML_vc = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="ML V/C"
            ).round(2)

            speed_ML = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Speed ML Real"
            ).round(2)

            ML_light_veh_period = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TransactionsDay_Lights"
            ).round(2)

            ML_MediumA_veh_period = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TransactionsDay_MediumA"
            ).round(2)

            ML_MediumB_veh_period = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TransactionsDay_MediumB"
            ).round(2)

            ML_light_veh_mile = df_filtered.pivot_table(
                index="Period",
                columns=["Direction", "Segment"],
                values="TransactionsLightsLength"
            ).round(2)

            ML_MediumA_veh_mile = df_filtered.pivot_table(
                index="Period",
                columns=["Direction", "Segment"],
                values="TransactionsMediumBLength"
            ).round(2)

            ML_MediumB_veh_mile = df_filtered.pivot_table(
                index="Period",
                columns=["Direction", "Segment"],
                values="TransactionsMediumALength"
            ).round(2)

            time_ML = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Time ML"
            ).round(2)

            # TBD: Toll bonus, reliability

            # Column 3

            GP_PCE_miles = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="GP PCE Total"
            ).round(2)

            GP_Volume = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="GP Volume"
            ).round(2)

            GP_flow = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="GP PCE"
            ).round(2)

            sum_GP_VC = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="GP V/C normal"
            ).round(2)

            sum_GP_VC_nominal = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="GP V/C capacity factors"
            ).round(2)

            speed_GP = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Speed GP Real"
            ).round(2)

            time_GP = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Time GP"
            ).round(2)

            # TBD: Toll bonus, reliability

            # Column 4

            corridor_PCE_total = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Highway PCE Total"
            ).round(2)

            corridor_volume = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Highway Volume"
            ).round(2)

            corridor_flow = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Highway PCE"
            ).round(2)

            corridor_vc = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Highway V/C"
            ).round(2)

            time_savings = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Time Savings"
            ).round(2)

            difference_time = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Difference in Travel Time"
            ).round(2)

            implied_vot = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="Implied Min VOT"
            ).round(2)

            toll_lights_seg = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TollLightsPerSeg"
            ).round(2)

            toll_mediumA_seg = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TollMediumAPerSeg"
            ).round(2)

            toll_mediumB_seg = df_filtered.pivot(
                index="Period",
                columns=["Direction", "Segment"],
                values="TollMediumBPerSeg"
            ).round(2)

            # sum_GPVeh = df_filtered.pivot(
            #     index="Period",
            #     columns=["Direction", "Segment"],
            #     values="GPVeh"
            # )

            # sum_MLVeh = df_filtered.pivot(
            #     index="Period",
            #     columns=["Direction", "Segment"],
            #     values="MLVeh"
            # )

            # sumRevHour = df_filtered.pivot(
            #     index="Period",
            #     columns=["Direction", "Segment"],
            #     values="RevenuePerHour"
            # )

            pivot_vals_column_1 = [sumTollLights, sumCaptureRateLights, revenuePerPeriod, revenuePerPeriod_LV, revenuePerPeriod_MediumA, revenuePerPeriod_MediumB, percentage_revenue]

            keys_1 = ["TollLights", "CaptureRateLights", "Revenue Per Period", "Revenue Per Period Lights", "Revenue Per Period Medium A", "Revenue Per Period Medium B", "% Workday Revenue"]
            
            pivot_vals_1 = dict(zip(keys_1, pivot_vals_column_1))

            pivot_vals_column_2 = [ML_PCE_miles, ML_Volume, ML_flow, ML_vc, speed_ML, time_ML, ML_light_veh_mile, ML_light_veh_period, ML_MediumA_veh_mile, ML_MediumA_veh_period, ML_MediumB_veh_mile, ML_MediumB_veh_period]
            
            keys_2 = ["ML PCE.Miles", "ML Volume (PCE/Period)", "ML Flow (PCE/hr)", "ML V/C", "ML Speed", "ML time", "ML light veh.Miles", "ML light veh/period", "ML medium A veh.Miles", "ML medium A veh/period", "ML medium B veh.Miles", "ML medium B veh/period"]

            pivot_vals_2 = dict(zip(keys_2, pivot_vals_column_2))
            
            pivot_vals_column_3 = [GP_PCE_miles, GP_Volume, GP_flow, sum_GP_VC, sum_GP_VC_nominal, speed_GP, time_GP]

            keys_3 = ["GP PCE.Miles", "GP Volume (PCE/Period)", "GP Flow (PCE/hr)", "GP V/C nominal", "GP V/C real", "GP Speed", "GP Time"]
            
            pivot_vals_3 = dict(zip(keys_3, pivot_vals_column_3))

            pivot_vals_column_4 = [corridor_PCE_total, corridor_volume, corridor_flow, corridor_vc, time_savings, difference_time, toll_lights_seg, toll_mediumA_seg, toll_mediumB_seg, implied_vot]

            keys_4 = ["Highway PCE.Miles", "Highway Volume", "Highway Flow", "Highway Vol / GP Capacity", "Time Saving", "Difference in travel time", "LVr Toll", "Medium A Toll", "Medium B Toll", "Implied VOT"]

            pivot_vals_4 = dict(zip(keys_4, pivot_vals_column_4))

            pd.DataFrame().to_excel(writer, sheet_name=f'Year_{year_val}')

            startcol = 1

            start_row = 0

            worksheet = writer.sheets[f'Year_{year_val}']

            worksheet.cell(start_row + 1, startcol + 1, "Lenght")

            styled_table = lengths

            # styled_table = styled_table.format("{:.2f}")

            styled_table.to_excel(writer, sheet_name=f'Year_{year_val}', startrow=start_row + 1, startcol=startcol)

            start_row += lengths.shape[0] + 3

            for key, pivot_vals in pivot_vals_1.items():

                worksheet = writer.sheets[f'Year_{year_val}']

                worksheet.cell(start_row + 1, startcol + 1, key)

                start_row += 1

                start_column_cumm = pivot_vals.shape[1] - 1

                styled_table = apply_light_green_style(pivot_vals)

                styled_table = styled_table.format("{:.2f}")

                styled_table.to_excel(writer, sheet_name=f'Year_{year_val}', startrow=start_row, startcol=startcol)

                start_row += pivot_vals.shape[0] + 5
            
            startcol += start_column_cumm + 3

            start_row = 4
        
            for key, pivot_vals in pivot_vals_2.items():

                worksheet = writer.sheets[f'Year_{year_val}']

                worksheet.cell(start_row + 1, startcol + 1, key)

                start_row += 1

                start_column_cumm = pivot_vals.shape[1]

                styled_table = apply_light_green_style(pivot_vals)

                styled_table = styled_table.format("{:.2f}")

                styled_table.to_excel(writer, sheet_name=f'Year_{year_val}', startrow=start_row, startcol=startcol)

                start_row += pivot_vals.shape[0] + 5
            
            startcol += start_column_cumm + 3

            start_row = 4

            for key, pivot_vals in pivot_vals_3.items():

                worksheet = writer.sheets[f'Year_{year_val}']

                worksheet.cell(start_row + 1, startcol + 1, key)

                start_row += 1

                start_column_cumm = pivot_vals.shape[1]

                styled_table = apply_light_green_style(pivot_vals)

                styled_table = styled_table.format("{:.2f}")

                styled_table.to_excel(writer, sheet_name=f'Year_{year_val}', startrow=start_row, startcol=startcol)

                start_row += pivot_vals.shape[0] + 5
            
            startcol += start_column_cumm + 3

            start_row = 4

            for key, pivot_vals in pivot_vals_4.items():

                worksheet = writer.sheets[f'Year_{year_val}']

                worksheet.cell(start_row + 1, startcol + 1, key)

                start_row += 1

                start_column_cumm = pivot_vals.shape[1]

                styled_table = apply_light_green_style(pivot_vals)

                styled_table = styled_table.format("{:.2f}")

                styled_table.to_excel(writer, sheet_name=f'Year_{year_val}', startrow=start_row, startcol=startcol)

                start_row += pivot_vals.shape[0] + 5
            
            startcol += start_column_cumm + 3

In [992]:
years = [2025,2032,2040,2050]

generate_pivot(first_model_df, f'{run_folder}/pivot.xlsx', years)

C:\Users\crisp\AppData\Local\Temp\ipykernel_24688\1729112946.py:15: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  lengths = df_filtered.pivot_table(
C:\Users\crisp\AppData\Local\Temp\ipykernel_24688\1729112946.py:114: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  ML_light_veh_mile = df_filtered.pivot_table(
C:\Users\crisp\AppData\Local\Temp\ipykernel_24688\1729112946.py:120: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  ML_MediumA_veh_mile = df_filtered.pivot_table(
C:\Users\crisp\AppData\Local\Temp\ipy

In [993]:
def generate_revenue_stream(final_df, file_name, both_dirs):

    if both_dirs:
        directions = 2
    else:
        directions = 1

    output_df = pd.DataFrame(columns=['Year', 'Total Revenue', 'Total Transactions', 'AADT', 'Toll/AADT/mi', 'Toll/AADT/mi Lights', 'Capture', 'GP Traffic', 'Corridor PCE']) # AADT refers to ML

    years = [2025, 2032, 2040, 2050]

    Anualization_factor = 308 # TBD: Just changed from 290, check if it makes sense # Try 308
    traffic_anualization = 315 # TBD: Check if it fits
    year_days = 365

    length_correction_factor = 1.038

    for year in years:

        yearly_df = final_df[final_df["Year"] == year]

        # We compute the revenues

        revenues_df = yearly_df.groupby(['Segment', 'Direction'])[["RevenuePerDay"]].sum().reset_index() #

        revenues_df["Annual Revenue"] = revenues_df["RevenuePerDay"] * Anualization_factor # We have to change 290 for an annual constant

        revenues_lights_df =  yearly_df.groupby(['Segment', 'Direction'])[["RevenuePerDayLights"]].sum().reset_index()

        revenues_lights_df["Annual Revenue"] = revenues_lights_df["RevenuePerDayLights"] * Anualization_factor

        total_revenues = revenues_df["Annual Revenue"].sum() * length_correction_factor # TBD: Change 1.02 for a parameter

        total_revenues_lights = revenues_lights_df["Annual Revenue"].sum() * length_correction_factor

        # We compute the total transactions

        transactions_df = yearly_df.groupby(['Segment', 'Direction'])[["TransactionsDay"]].sum().reset_index()

        # transactions_df_aux.append(transactions_df)

        total_transactions = transactions_df["TransactionsDay"].sum() * traffic_anualization # TBD: Change 315 for a parameter

        # We compute the AADT

        aadt_df = yearly_df.groupby(['SegDir'])[["TransactionsDay"]].sum()

        aadt_lights_df = yearly_df.groupby(['SegDir'])[["TransactionsDay_Lights"]].sum()

        gp_traffic_df = yearly_df.groupby(['SegDir'])[["GPVehDay"]].sum()

        corridor_df = yearly_df.groupby(['SegDir'])[["Highway Volume"]].sum()

        df_lengths = pd.DataFrame(seg_params["Length"])

        merged_df = pd.merge(aadt_df, df_lengths, on='SegDir')

        merged_df_lights = pd.merge(aadt_lights_df, df_lengths, on='SegDir')

        merged_gp_traffic_df = pd.merge(gp_traffic_df, df_lengths, on='SegDir')

        merged_corridor_df = pd.merge(corridor_df, df_lengths, on='SegDir')

        # Compute product
        merged_df['Weighted'] = merged_df['Length'].values * merged_df["TransactionsDay"].values

        merged_df_lights['Weighted'] = merged_df_lights['Length'].values * merged_df_lights["TransactionsDay_Lights"].values

        merged_gp_traffic_df['Weighted'] = merged_gp_traffic_df['Length'] * merged_gp_traffic_df["GPVehDay"]

        merged_corridor_df['Weighted'] = merged_corridor_df['Length'] * merged_corridor_df["Highway Volume"]

        # Sum weighted values
        total_aadt = (traffic_anualization/year_days) * merged_df['Weighted'].sum() / (merged_df['Length'].sum()/directions)

        total_ligths_aadt = (traffic_anualization/year_days) * merged_df_lights['Weighted'].sum() / (merged_df_lights['Length'].sum()/directions)

        traffic_gp = (traffic_anualization/year_days) * merged_gp_traffic_df['Weighted'].sum() / (merged_gp_traffic_df['Length'].sum()/directions)

        gp_ml = total_aadt + traffic_gp

        corridor_pce = (traffic_anualization/year_days) * merged_corridor_df['Weighted'].sum() / (merged_corridor_df['Length'].sum()/directions)

        capture_total = total_aadt / gp_ml

        toll_AADT_mi = total_revenues / (year_days * total_aadt * (merged_df['Length'].sum()/directions))

        toll_AADT_mi_lights = total_revenues_lights / (year_days * total_ligths_aadt * (merged_df_lights['Length'].sum()/directions))
        
        output_df.loc[len(output_df)] = [year, total_revenues, total_transactions, total_aadt, toll_AADT_mi, toll_AADT_mi_lights, capture_total, traffic_gp, corridor_pce]

    # Define full range of years to interpolate over
    full_years = pd.DataFrame({"Year": np.arange(int(output_df["Year"].min()), int(output_df["Year"].max()) + 1)})

    # Merge with original to create missing years with NaNs
    merged = pd.merge(full_years, output_df, on="Year", how="left")

    # Interpolate all numeric columns except "Year"
    interpolated = np.log(merged).interpolate(method="linear")

    interpolated = np.exp(interpolated)

    # Copy original column names
    original_columns = list(interpolated.columns)

    # Track how many columns we've inserted to adjust the index
    insert_count = 0

    # Start from index 1 to skip the first column (index 0)
    for i in range(1, len(original_columns)):
        col = original_columns[i]
        new_col_name = f"Var {col} (%)"
        insert_position = i + insert_count + 1  # Adjust position with insert_count
        interpolated[col] = interpolated[col].round(6)
        new_col_val = interpolated[col].pct_change()
        interpolated.insert(insert_position, new_col_name, new_col_val)
        insert_count += 1

    interpolated.to_csv(file_name)

    return interpolated


In [994]:
# Here we filter which section of the df we compute

subphase_1_segdir = ['1WB','2WB','3WB','4WB','5WB','10WB']

subphase_2_segdir = ['1EB','2EB','3EB','4EB','5EB','10EB']

subphase_12_segdir = subphase_1_segdir + subphase_2_segdir

subphase_extensions_segdir = ['6EB','11WB','13EB'] #['6WB','11WB','13WB','6EB','11EB','13EB']

# subphase_a_segdir = ['1EB','2EB']

# subphase_b_segdir = ['3EB','4EB','5EB']

# subphase_c_segdir = ['6EB','7EB']

# subphase_d_segdir = ['8EB','9EB']

# subphase_e_segdir = ['6WB','7WB']

# subphase_f_segdir = ['8WB','9WB']

subphase_6_segdir = ['6WB']
# subphase_7_segdir = ['7WB','7EB']
# subphase_8_segdir = ['8WB','8EB']
# subphase_9_segdir = ['9WB','9EB']
subphase_SR400_segdir = ['11WB']
subphase_PIB_segdir = ['12WB','12EB']
subphase_I85_segdir = ['13EB']

subphase_1 = first_model_df[first_model_df['SegDir'].isin(subphase_1_segdir)]

subphase_2 = first_model_df[first_model_df['SegDir'].isin(subphase_2_segdir)]

subphase_12 = first_model_df[first_model_df['SegDir'].isin(subphase_12_segdir)]

subphase_extensions = first_model_df[first_model_df['SegDir'].isin(subphase_extensions_segdir)]

subphase_s6 = first_model_df[first_model_df['SegDir'].isin(subphase_6_segdir)]

# subphase_s7 = first_model_df[first_model_df['SegDir'].isin(subphase_7_segdir)]

# subphase_s8 = first_model_df[first_model_df['SegDir'].isin(subphase_8_segdir)]

# subphase_s9 = first_model_df[first_model_df['SegDir'].isin(subphase_9_segdir)]

subphase_SR400 = first_model_df[first_model_df['SegDir'].isin(subphase_SR400_segdir)]

subphase_PIB = first_model_df[first_model_df['SegDir'].isin(subphase_PIB_segdir)]

subphase_I85 = first_model_df[first_model_df['SegDir'].isin(subphase_I85_segdir)]

interpolated = generate_revenue_stream(first_model_df, f'{run_folder}/revenue_stream.csv', True)

revenue_stream_subphase_1 = generate_revenue_stream(subphase_1, f'{run_folder}/revenue_stream_phase1.csv', False)

revenue_stream_subphase_2 = generate_revenue_stream(subphase_2, f'{run_folder}/revenue_stream_phase2.csv', False)

revenue_stream_subphase_12 = generate_revenue_stream(subphase_12, f'{run_folder}/revenue_stream_phase12.csv', False)

revenue_stream_extensions = generate_revenue_stream(subphase_extensions, f'{run_folder}/revenue_stream_extensions.csv', False)

revenue_stream_subphase_6 = generate_revenue_stream(subphase_s6, f'{run_folder}/revenue_stream_phase6.csv', False)

# revenue_stream_subphase_7 = generate_revenue_stream(subphase_s7, f'{run_folder}/revenue_stream_phase7.csv', True)

# revenue_stream_subphase_8 = generate_revenue_stream(subphase_s8, f'{run_folder}/revenue_stream_phase8.csv', True)

# revenue_stream_subphase_9 = generate_revenue_stream(subphase_s9, f'{run_folder}/revenue_stream_phase9.csv', True)

revenue_stream_subphase_SR400 = generate_revenue_stream(subphase_SR400, f'{run_folder}/revenue_stream_phaseSR400.csv', False)

revenue_stream_subphase_PIB = generate_revenue_stream(subphase_PIB, f'{run_folder}/revenue_stream_phasePIB.csv', False)

revenue_stream_subphase_I85 = generate_revenue_stream(subphase_I85, f'{run_folder}/revenue_stream_phaseI85.csv', True)

# interpolated = revenue_stream_subphase_1

revenue_stream_subphase_12

,Year,Total Revenue,Var Total Revenue (%),Total Transactions,Var Total Transactions (%),AADT,Var AADT (%),Toll/AADT/mi,Var Toll/AADT/mi (%),Toll/AADT/mi Lights,Var Toll/AADT/mi Lights (%),Capture,Var Capture (%),GP Traffic,Var GP Traffic (%),Corridor PCE,Var Corridor PCE (%)
0,2025.0,4.393290e+08,NaN,6.877373e+07,NaN,15540.892580,NaN,2.116118,NaN,1.902253,NaN,0.174736,NaN,73398.546571,NaN,104773.693962,NaN
1,2026.0,4.875002e+08,0.109647,7.294868e+07,0.060706,16432.855789,0.057395,2.220689,0.049416,1.998630,0.050665,0.178445,0.021226,75628.338597,0.030379,108365.800252,0.034284
2,2027.0,5.409532e+08,0.109647,7.737708e+07,0.060706,17376.012864,0.057395,2.330428,0.049417,2.099889,0.050664,0.182234,0.021233,77925.870009,0.030379,112081.059855,0.034284
3,2028.0,6.002672e+08,0.109647,8.207431e+07,0.060706,18373.302056,0.057395,2.445589,0.049416,2.206278,0.050664,0.186102,0.021225,80293.198678,0.030379,115923.695012,0.034284
4,2029.0,6.660848e+08,0.109647,8.705669e+07,0.060706,19427.830255,0.057395,2.566441,0.049416,2.318058,0.050665,0.190053,0.021230,82732.444991,0.030379,119898.072723,0.034284
5,2030.0,7.391191e+08,0.109647,9.234153e+07,0.060706,20542.882672,0.057395,2.693265,0.049416,2.435500,0.050664,0.194088,0.021231,85245.793752,0.030379,124008.709704,0.034284
6,2031.0,8.201614e+08,0.109647,9.794719e+07,0.060706,21721.933068,0.057395,2.826356,0.049416,2.558893,0.050664,0.198208,0.021227,87835.496138,0.030379,128260.277528,0.034284
7,2032.0,9.100898e+08,0.109647,1.038931e+08,0.060706,22968.654582,0.057395,2.966025,0.049417,2.688538,0.050664,0.202416,0.021230,90503.871712,0.030379,132657.607928,0.034284
8,2033.0,9.915019e+08,0.089455,1.063483e+08,0.023631,23483.244812,0.022404,3.160542,0.065582,2.864937,0.065611,0.203429,0.005005,91951.235579,0.015992,135050.697559,0.018040
9,2034.0,1.080197e+09,0.089455,1.088614e+08,0.023631,24009.363932,0.022404,3.367815,0.065581,3.052911,0.065612,0.204447,0.005004,93421.746105,0.015992,137486.957559,0.018040
